# HW4 – Residual Analysis and K-Means Clustering
**Alina’s homework 4 code   ———— Authors: alina & chatgpt**
## Question 1: Clustering Store Performance

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

df = pd.read_csv("/Users/anxinyi/Documents/GitHub/AAE625_2026/RetailData_A202602.csv")

# Log variables
df['ln_Q'] = np.log(df['QUANTITY'])
df['ln_P'] = np.log(df['PRICE'])
df['ln_I'] = np.log(df['AVERAGE_INCOME'])
df['ln_R'] = np.log(df['PERCENTAGE_OF_RENTERS'])
df['ln_C'] = np.log(df['PERCENT_HAVING_CHILDREN'])
df['ln_S'] = np.log(df['PERCENT_SPEAKING_SPANISH'])

# OLS
X = df[['ln_P','ln_I','ln_R','ln_C','ln_S']]
X = sm.add_constant(X)
y = df['ln_Q']

model = sm.OLS(y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                   ln_Q   R-squared:                       0.665
Model:                            OLS   Adj. R-squared:                  0.659
Method:                 Least Squares   F-statistic:                     113.1
Date:                Wed, 22 Apr 2026   Prob (F-statistic):           1.47e-65
Time:                        15:18:42   Log-Likelihood:                -30.391
No. Observations:                 291   AIC:                             72.78
Df Residuals:                     285   BIC:                             94.82
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          8.7401      0.706     12.373      0.0

In [2]:
# Residual construction
df['ln_Q_hat'] = model.predict(X)
df['Q_hat'] = np.exp(df['ln_Q_hat'])

df['R_QUANTITY'] = df['QUANTITY'] - df['Q_hat']
df['PRICE_DELTA'] = df['PRICE'] - df['PRICE'].mean()

### (a) Standardization and K-Means Clustering

In [3]:
features = df[['PRICE_DELTA','R_QUANTITY']]

scaler = StandardScaler()
scaled = scaler.fit_transform(features)

kmeans = KMeans(n_clusters=5, n_init=10, random_state=42)
kmeans.fit(scaled)

print("Cluster Centers (standardized):\n", kmeans.cluster_centers_)
print("Iterations:", kmeans.n_iter_)

Cluster Centers (standardized):
 [[-1.50999433 -0.42802311]
 [ 0.1641006   0.2081588 ]
 [ 0.06437096 -1.34762709]
 [ 1.74172074 -0.53734294]
 [-0.10691312  1.59464397]]
Iterations: 10


### (b) Scatter Plot

In [4]:
df['CLUSTER'] = kmeans.labels_.astype(str)

fig = px.scatter(
    df,
    x='R_QUANTITY',
    y='PRICE_DELTA',
    color='CLUSTER',
    labels={
        "R_QUANTITY": "Residual Quantity (Performance)",
        "PRICE_DELTA": "Price Deviation"
    },
    title='R_QUANTITY vs PRICE_DELTA by Cluster'
)

fig.show()

### (c) Cluster Summary and Economic Interpretation

In [5]:
summary = df.groupby('CLUSTER')[['PRICE_DELTA','R_QUANTITY']].mean()
summary

,PRICE_DELTA,R_QUANTITY
CLUSTER,,
0,-0.721092,-3881.834450
1,0.078366,4535.740112
2,0.030740,-16049.478683
3,0.831752,-5328.288362
4,-0.051056,22880.874082


**Cluster Interpretation:**

- Cluster 0: Slightly high price & over-performing → successful premium pricing strategy  
- Cluster 1: Slightly high price & under-performing → potential overpricing  
- Cluster 2: Very high price & under-performing → clearly overpriced stores losing customers  
- Cluster 3: Low price & under-performing → weak demand or operational inefficiency  
- Cluster 4: Low price & over-performing → underpriced stores with strong demand  

**Direct Answers to the Question:**

- Clusters representing **low price & over-performing stores**: Cluster 4  
- Clusters representing **high price & under-performing stores**: Clusters 1 and 2  

These results reveal clear pricing-performance tradeoffs and identify opportunities for pricing optimization.
